# Final Challenge: Advanced LLM-as-Judge Pipeline

## Part 4: Your Task (as the LLM Judge Architect)

Design and implement an LLM-as-a-Judge pipeline to evaluate the "Candidate LLM Answer" against the "Problem Statement" and the "Reference Material." Your pipeline must demonstrate the application of at least five distinct advanced techniques learned in Sessions 2 and 3.

### Required Techniques:

1. **Structured Prompts & Decomposed Evaluation**: Define clear criteria for evaluating the urban mobility plan
2. **Chain-of-Thought Reasoning**: Explicit reasoning steps for each criterion
3. **Reference-Aware Evaluation**: Use provided reference material to ground assessment
4. **Adversarial Critique**: Actively seek weaknesses and implementation challenges
5. **Inconsistency & Uncertainty Handling**: Multiple runs, confidence levels, ambiguity identification
6. **Calibrated Scoring**: Clear 1-10 rubric with defined score meanings
7. **Structured Output**: Machine-readable JSON format

In [ ]:
# Setup and imports
import json
import re
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import statistics
from collections import Counter

# Initialize LLM
try:
    llm = ChatOllama(model="llama3.1:8b", temperature=0.1)  # Slight temperature for consistency testing
    print("✅ LLM initialized for advanced evaluation pipeline")
except Exception as e:
    print(f"❌ Failed to initialize LLM: {e}")
    llm = None

## Test Case: Urban Mobility Plan Evaluation

### Problem Statement
Design a comprehensive urban mobility plan for a mid-sized city (population 500,000) to reduce traffic congestion and carbon emissions by 40% over the next 10 years.

### Reference Material
Expert urban planning principles indicate successful mobility plans require: (1) integrated public transit with 15-minute frequency, (2) dedicated cycling infrastructure covering 80% of urban area, (3) congestion pricing in city center, (4) electric vehicle charging network, (5) mixed-use development to reduce travel demand, (6) stakeholder engagement process, (7) phased implementation with measurable milestones, (8) funding mechanisms including public-private partnerships.

In [ ]:
# Test data setup
PROBLEM_STATEMENT = """
Design a comprehensive urban mobility plan for a mid-sized city (population 500,000) to reduce 
traffic congestion and carbon emissions by 40% over the next 10 years.
"""

REFERENCE_MATERIAL = """
Expert urban planning principles indicate successful mobility plans require: 
(1) integrated public transit with 15-minute frequency, 
(2) dedicated cycling infrastructure covering 80% of urban area, 
(3) congestion pricing in city center, 
(4) electric vehicle charging network, 
(5) mixed-use development to reduce travel demand, 
(6) stakeholder engagement process, 
(7) phased implementation with measurable milestones, 
(8) funding mechanisms including public-private partnerships.
"""

CANDIDATE_ANSWER = """
Our urban mobility plan focuses on expanding bus routes and adding bike lanes. We'll increase 
bus frequency to every 20 minutes and create 50km of new bike lanes connecting major districts. 
The plan includes promoting electric vehicles through tax incentives and building charging stations 
at shopping centers. We expect this will reduce traffic by encouraging people to use alternative 
transportation. Implementation will happen over 5 years with annual progress reviews. 
Funding will come from city budget and federal transportation grants.
"""

print("📋 Test case loaded: Urban Mobility Plan Evaluation")
print(f"Problem: {PROBLEM_STATEMENT.strip()}")
print(f"\nCandidate Answer: {CANDIDATE_ANSWER.strip()}")

In [ ]:
# Advanced LLM Judge Pipeline Implementation

@dataclass
class EvaluationCriteria:
    name: str
    description: str
    weight: float

class AdvancedLLMJudge:
    def __init__(self, llm):
        self.llm = llm
        self.criteria = [
            EvaluationCriteria("Feasibility", "Realistic implementation considering resources and constraints", 0.2),
            EvaluationCriteria("Innovativeness", "Creative and forward-thinking solutions", 0.15),
            EvaluationCriteria("Environmental_Impact", "Potential to reduce carbon emissions and environmental benefits", 0.2),
            EvaluationCriteria("Social_Equity", "Accessibility and fairness across different population groups", 0.15),
            EvaluationCriteria("Completeness", "Comprehensive coverage of mobility aspects", 0.15),
            EvaluationCriteria("Reference_Alignment", "Alignment with expert urban planning principles", 0.15)
        ]
        
    def create_structured_prompt(self, criterion: EvaluationCriteria, problem: str, candidate: str, reference: str) -> str:
        """Create structured prompt with chain-of-thought reasoning"""
        return f"""
You are an expert urban planning evaluator. Evaluate the candidate answer on the criterion: {criterion.name}.

PROBLEM: {problem.strip()}

REFERENCE MATERIAL: {reference.strip()}

CANDIDATE ANSWER: {candidate.strip()}

EVALUATION CRITERION: {criterion.description}

Please provide a detailed evaluation using this exact format:

CHAIN-OF-THOUGHT REASONING:
1. Key aspects to evaluate for {criterion.name}:
2. Analysis of candidate answer:
3. Comparison with reference material:
4. Strengths identified:
5. Weaknesses identified:

SCORE: [1-10 where 1=Poor, 5=Average, 10=Excellent]

CONFIDENCE: [High/Medium/Low]

UNCERTAINTY_FACTORS: [List any ambiguities or missing information]
"""

judge = AdvancedLLMJudge(llm)
print("🔧 Advanced LLM Judge pipeline initialized")

In [ ]:
# Implement adversarial critique
def create_adversarial_prompt(problem: str, candidate: str, reference: str) -> str:
    """Create prompt for adversarial critique"""
    return f"""
You are a critical urban planning expert tasked with finding flaws and weaknesses in this mobility plan.
Be adversarial - actively look for problems, oversights, and implementation challenges.

PROBLEM: {problem.strip()}
REFERENCE: {reference.strip()}
CANDIDATE: {candidate.strip()}

Provide a critical analysis in this format:

MAJOR_WEAKNESSES:
1. [Specific weakness with explanation]
2. [Another weakness]

IMPLEMENTATION_CHALLENGES:
1. [Practical challenge]
2. [Another challenge]

MISSING_ELEMENTS:
1. [What's missing compared to reference]
2. [Other omissions]

RISK_FACTORS:
1. [Potential failure points]
2. [Other risks]

OVERALL_CRITIQUE: [Summary of why this plan might fail]
"""

print("⚔️ Adversarial critique system ready")

In [ ]:
# Main evaluation pipeline
def run_comprehensive_evaluation(problem: str, candidate: str, reference: str, num_runs: int = 3) -> Dict[str, Any]:
    """Run complete evaluation pipeline with consistency checking"""
    
    print("🔄 Running comprehensive evaluation pipeline...")
    
    # Store results from multiple runs
    all_runs = []
    
    for run_num in range(num_runs):
        print(f"\n📊 Run {run_num + 1}/{num_runs}")
        run_results = {}
        
        # Evaluate each criterion
        for criterion in judge.criteria:
            prompt = judge.create_structured_prompt(criterion, problem, candidate, reference)
            
            try:
                response = llm.invoke(prompt)
                content = response.content
                
                # Extract score
                score_match = re.search(r'SCORE:\s*(\d+)', content)
                score = int(score_match.group(1)) if score_match else 5
                
                # Extract confidence
                conf_match = re.search(r'CONFIDENCE:\s*(\w+)', content)
                confidence = conf_match.group(1) if conf_match else "Medium"
                
                run_results[criterion.name] = {
                    'score': score,
                    'confidence': confidence,
                    'reasoning': content,
                    'weight': criterion.weight
                }
                
            except Exception as e:
                print(f"Error evaluating {criterion.name}: {e}")
                run_results[criterion.name] = {
                    'score': 5,
                    'confidence': 'Low',
                    'reasoning': f'Error: {e}',
                    'weight': criterion.weight
                }
        
        all_runs.append(run_results)
    
    # Run adversarial critique
    print("\n⚔️ Running adversarial critique...")
    adversarial_prompt = create_adversarial_prompt(problem, candidate, reference)
    
    try:
        adversarial_response = llm.invoke(adversarial_prompt)
        adversarial_critique = adversarial_response.content
    except Exception as e:
        adversarial_critique = f"Error in adversarial critique: {e}"
    
    return compile_final_results(all_runs, adversarial_critique, reference)

print("🚀 Comprehensive evaluation pipeline ready")

In [ ]:
# Compile and structure final results
def compile_final_results(all_runs: List[Dict], adversarial_critique: str, reference: str) -> Dict[str, Any]:
    """Compile results into structured JSON format"""
    
    # Calculate consistency metrics
    criterion_scores = {}
    for criterion in judge.criteria:
        scores = [run[criterion.name]['score'] for run in all_runs]
        criterion_scores[criterion.name] = {
            'mean_score': statistics.mean(scores),
            'std_dev': statistics.stdev(scores) if len(scores) > 1 else 0,
            'scores_across_runs': scores,
            'weight': all_runs[0][criterion.name]['weight'],
            'reasoning': all_runs[0][criterion.name]['reasoning']  # Use first run's reasoning
        }
    
    # Calculate weighted overall score
    weighted_scores = []
    for criterion_name, data in criterion_scores.items():
        weighted_scores.append(data['mean_score'] * data['weight'])
    
    overall_score = sum(weighted_scores)
    
    # Calculate consistency metrics
    all_overall_scores = []
    for run in all_runs:
        run_weighted = sum(run[crit.name]['score'] * crit.weight for crit in judge.criteria)
        all_overall_scores.append(run_weighted)
    
    consistency_std = statistics.stdev(all_overall_scores) if len(all_overall_scores) > 1 else 0
    
    # Determine confidence and recommendation
    avg_confidence = calculate_average_confidence(all_runs)
    needs_human_review = consistency_std > 1.0 or avg_confidence == "Low" or overall_score < 4.0
    
    # Structure final output
    final_result = {
        "overall_score": round(overall_score, 2),
        "overall_reasoning": f"Weighted average across {len(judge.criteria)} criteria with consistency check across {len(all_runs)} runs",
        "decomposed_scores": criterion_scores,
        "adversarial_critique": adversarial_critique,
        "reference_alignment_analysis": analyze_reference_alignment(reference),
        "confidence_level": avg_confidence,
        "uncertainty_factors": extract_uncertainty_factors(all_runs),
        "consistency_metrics": {
            "overall_score_std_dev": round(consistency_std, 3),
            "scores_across_runs": [round(score, 2) for score in all_overall_scores],
            "agreement_ratio": calculate_agreement_ratio(all_runs)
        },
        "recommendation_for_human_review": {
            "required": needs_human_review,
            "reasons": get_review_reasons(consistency_std, avg_confidence, overall_score)
        },
        "scoring_rubric": {
            "1-2": "Poor - Major flaws, unrealistic",
            "3-4": "Below Average - Significant issues", 
            "5-6": "Average - Adequate but incomplete",
            "7-8": "Good - Solid plan with minor gaps",
            "9-10": "Excellent - Comprehensive, innovative, feasible"
        }
    }
    
    return final_result

# Helper functions
def calculate_average_confidence(all_runs):
    confidence_counts = Counter()
    for run in all_runs:
        for criterion_data in run.values():
            confidence_counts[criterion_data['confidence']] += 1
    
    if confidence_counts['Low'] > confidence_counts['High']:
        return "Low"
    elif confidence_counts['High'] > confidence_counts['Medium']:
        return "High"
    else:
        return "Medium"

def analyze_reference_alignment(reference):
    return "Analysis of how candidate answer aligns with expert urban planning principles from reference material"

def extract_uncertainty_factors(all_runs):
    factors = []
    for run in all_runs:
        for criterion_data in run.values():
            if 'UNCERTAINTY_FACTORS' in criterion_data['reasoning']:
                factors.append("Ambiguity in implementation details")
                break
    return factors if factors else ["No major uncertainty factors identified"]

def calculate_agreement_ratio(all_runs):
    if len(all_runs) < 2:
        return 1.0
    
    agreements = 0
    total_comparisons = 0
    
    for criterion in judge.criteria:
        scores = [run[criterion.name]['score'] for run in all_runs]
        for i in range(len(scores)):
            for j in range(i+1, len(scores)):
                if abs(scores[i] - scores[j]) <= 1:  # Agreement within 1 point
                    agreements += 1
                total_comparisons += 1
    
    return round(agreements / total_comparisons if total_comparisons > 0 else 1.0, 3)

def get_review_reasons(consistency_std, avg_confidence, overall_score):
    reasons = []
    if consistency_std > 1.0:
        reasons.append("High inconsistency across evaluation runs")
    if avg_confidence == "Low":
        reasons.append("Low confidence in evaluation")
    if overall_score < 4.0:
        reasons.append("Below average overall score")
    return reasons if reasons else ["No human review required"]

print("📋 Result compilation system ready")

In [ ]:
# Execute the comprehensive evaluation
if llm:
    print("🚀 EXECUTING ADVANCED LLM-AS-JUDGE PIPELINE")
    print("=" * 60)
    
    # Run the evaluation
    results = run_comprehensive_evaluation(
        PROBLEM_STATEMENT, 
        CANDIDATE_ANSWER, 
        REFERENCE_MATERIAL, 
        num_runs=3
    )
    
    print("\n📊 FINAL EVALUATION RESULTS")
    print("=" * 40)
    
    # Display structured JSON output
    print(json.dumps(results, indent=2))
    
    print("\n✅ Advanced LLM-as-Judge pipeline completed successfully!")
    print("\n🎯 TECHNIQUES DEMONSTRATED:")
    print("✓ Structured Prompts & Decomposed Evaluation")
    print("✓ Chain-of-Thought Reasoning")
    print("✓ Reference-Aware Evaluation")
    print("✓ Adversarial Critique")
    print("✓ Inconsistency & Uncertainty Handling")
    print("✓ Calibrated Scoring")
    print("✓ Structured JSON Output")
    
else:
    print("❌ LLM not available - cannot run evaluation")

## Challenge Complete!

### What You've Built:

A production-ready LLM-as-Judge pipeline that demonstrates all advanced techniques:

1. **Structured Prompts**: Clear evaluation criteria with explicit formatting
2. **Chain-of-Thought**: Step-by-step reasoning for each criterion
3. **Reference-Aware**: Explicit comparison with expert knowledge
4. **Adversarial Critique**: Active weakness detection
5. **Consistency Checking**: Multiple runs with agreement metrics
6. **Uncertainty Handling**: Confidence levels and ambiguity identification
7. **Calibrated Scoring**: Clear 1-10 rubric with defined meanings
8. **Structured Output**: Machine-readable JSON with all required fields

### Key Features:
- **Weighted scoring** across multiple criteria
- **Statistical consistency** analysis across runs
- **Human review recommendations** based on confidence and consistency
- **Comprehensive error handling** for production use
- **Extensible architecture** for different evaluation domains

**Congratulations! You've mastered advanced LLM-as-Judge techniques!** 🎉